# LCA Lasers4MaaS 



In [ ]:
import bw2data as bd
import bw2io as bi
from bw2calc import LCA
import pandas as pd
import ipywidgets as widgets
import numpy as np
from IPython.display import display

In [ ]:
PROJECT_NAME = "Lasers4MaaS"
ECOINVENT_DATABASE_NAME = "ecoinvent-3.12-cutoff"

bd.projects.set_current(PROJECT_NAME)
if ECOINVENT_DATABASE_NAME not in bd.databases:
    bi.import_ecoinvent_release(version="3.12", system_model="cutoff")
else:
    print(f"Database {ECOINVENT_DATABASE_NAME} already exists, skipping import.")

In [ ]:
ei = bd.Database(ECOINVENT_DATABASE_NAME)

In [ ]:
bio = bd.Database("ecoinvent-3.12-biosphere")

### Global parameters

In [ ]:
# Technosphere exchanges
elec = [a for a in ei.search("market group for electricity, medium voltage") if a.get('location') == "RER"][0]
# wire = [a for a in ei.search("welding wire") if a.get('location') == "GLO"] # a proxy needs to be found
gas  = [a for a in ei.search("market for argon") if a.get('location') == "RER"][0]

# Biosphere exchanges
pm25 = bio.search("Particulate matter, < 2.5 um")[0]

In [ ]:
# Cost factors for the main resource inputs (adjust as needed)
COST_FACTORS = {
    'electricity_EUR_per_kWh': 0.29,     # EUR 0.29/kWh for industrial electricity
    'argon_EUR_per_kg': 5.87,           # EUR 2.5/kg for argon gas
    'nitrogen_EUR_per_kg': 5.35           # EUR 1.2/kg for nitrogen gas
}

In [ ]:
# Material assignments for each product (based on typical use cases)
MATERIAL_ASSIGNMENTS = {
    "Heat exchanger": "aluminium",      # Aluminium for heat exchangers
    "Fuel tank": "steel",             # Steel for fuel tanks
    "Heterogeneous pipe": "steel",    # Steel pipes
    "Freeze Dryers Shelf": "steel",   # Stainless steel for freeze dryers
    "Battery cooling plate": "aluminium"  # Aluminium for battery cooling
}

In [ ]:
# Get only the most used EF 3.1 methods
most_used_indicators = [
    'climate change: total',
    'particulate matter formation',
    'acidification',
    'eutrophication: terrestrial',
    'photochemical oxidant formation: human health',
    'human toxicity: carcinogenic'
]

ef31_methods = [
    m for m in bd.methods
    if ('EF v3.1' in str(m) or 'EF 3.1' in str(m))
    and 'no LT' not in str(m)
    and any(indicator in str(m) for indicator in most_used_indicators)
]

# Physical unit of each impact category, as stored in the method metadata
method_units = {m[2]: bd.Method(m).metadata.get("unit", "") for m in ef31_methods}

print(f"Found {len(ef31_methods)} most-used EF 3.1 methods")
print("Selected EF 3.1 impact categories:")
for m in ef31_methods:
    print(f"  {m}  ->  unit: {method_units[m[2]]}")

### Calculation engine for joining process

In [ ]:
def create_joining_exchanges(database_name, joining_params, product_name, elec_key, gas_key):
    """
    Create exchanges from joining parameters.
    minimum = lowest power (or gas flow) during the shortest joining time (fastest speed).
    maximum = highest power (or gas flow) during the longest joining time (slowest speed).
    amount = (minimum + maximum) / 2.
    """
    params = joining_params[product_name]

    # Joining time in seconds (length / speed): the fastest speed gives the shortest time
    time_min_s = params["joint_length_m"] / params["joining_speed_m_per_s"]["max"]
    time_max_s = params["joint_length_m"] / params["joining_speed_m_per_s"]["min"]

    # Electricity in kWh: (joining power + welding station power) in kW * hours
    power_min = params["power_kW_joining"]["min"] + params["power_kW_station"]["min"]
    power_max = params["power_kW_joining"]["max"] + params["power_kW_station"]["max"]
    elec_min = power_min * time_min_s / 3600
    elec_max = power_max * time_max_s / 3600
    elec_amount = (elec_min + elec_max) / 2

    # Gas in kg: gas_flow (kg/s) * time (s)
    gas_min = params["gas_flow_kg_per_s"]["min"] * time_min_s
    gas_max = params["gas_flow_kg_per_s"]["max"] * time_max_s
    gas_amount = (gas_min + gas_max) / 2

    exchanges = [
        {"input": (database_name, product_name), "amount": 1, "type": "production"},
    ]

    # Electricity exchange
    if elec_min != elec_max:
        exchanges.append({
            "input": elec_key,
            "amount": elec_amount,
            "minimum": elec_min,
            "maximum": elec_max,
            "uncertainty_type": 1,
            "type": "technosphere",
        })
    else:
        exchanges.append({
            "input": elec_key,
            "amount": elec_amount,
            "type": "technosphere",
        })

    # Gas exchange
    if gas_min != gas_max:
        exchanges.append({
            "input": gas_key,
            "amount": gas_amount,
            "minimum": gas_min,
            "maximum": gas_max,
            "uncertainty_type": 1,
            "type": "technosphere",
        })
    else:
        exchanges.append({
            "input": gas_key,
            "amount": gas_amount,
            "type": "technosphere",
        })
    return exchanges

In [ ]:
def calculate_process_cost(process, scenario="amount", cost_factors=None):
    """Calculate total cost for a process based on exchange amounts."""
    if cost_factors is None:
        cost_factors = COST_FACTORS

    total_cost = 0.0

    for exchange in process.exchanges():
        if exchange["type"] == "technosphere":
            # Get amount for chosen scenario
            if scenario == "minimum":
                amount = exchange.get("minimum", exchange.get("min", exchange["amount"]))
            elif scenario == "maximum":
                amount = exchange.get("maximum", exchange.get("max", exchange["amount"]))
            else:
                amount = exchange["amount"]

            # Identify resource type and calculate cost
            input_node = exchange.input
            input_str = str(input_node).lower()

            if "electricity" in input_str:
                total_cost += amount * cost_factors.get('electricity_EUR_per_kWh', 0.29)
            elif "argon" in input_str:
                total_cost += amount * cost_factors.get('argon_EUR_per_kg', 5.87)
            elif "nitrogen" in input_str:
                total_cost += amount * cost_factors.get('nitrogen_EUR_per_kg', 5.35)
    return total_cost

## Use case unit processes with State of the Art technologies

The functional unit is fixed at `1 joined unit` (`1 unit`).

In [ ]:
SoA_NAME = "State_of_the_Art"

SoA = bd.Database(SoA_NAME)

In [ ]:
# Joining parameters for SoA processes
# Define joining power and welding station power (kW), joining speed (m/s), joint length (m), and gas flow (kg/s) ranges
SoA_joining_params = {
    "Heat exchanger": {
        "power_kW_joining": {"min": 1.4, "max": 1.5},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.012, "max": 0.014},
        "joint_length_m": 42.9,
        "gas_flow_kg_per_s": {"min": 0.002676, "max": 0.002676},
        "time_m": {"min": 55, "max": 55}
    },
    "Fuel tank": {
        "power_kW_joining": {"min": 1.8, "max": 2.52},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.0025, "max": 0.0025},
        "joint_length_m": 5.06,
        "gas_flow_kg_per_s": {"min": 0.000268, "max": 0.000268},
        "time_m": {"min": 20, "max": 20}
    },
    "Heterogeneous pipe": {
        "power_kW_joining": {"min": 1.4, "max": 2.1},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.015, "max": 0.025},
        "joint_length_m": 0.094,
        "gas_flow_kg_per_s": {"min": 0.000892, "max": 0.000892},
        "time_m": {"min": 0.5, "max": 0.5}
    },
    "Freeze Dryers Shelf": {
        "power_kW_joining": {"min": 1.8, "max": 2.0},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.009, "max": 0.009},
        "joint_length_m": 13.2,
        "gas_flow_kg_per_s": {"min": 0.001487, "max": 0.001487},
        "time_m": {"min": 10, "max": 10}
    },
    "Battery cooling plate": {
        "power_kW_joining": {"min": 0, "max": 0},
        "power_kW_station": {"min": 2000, "max": 2000},
        "joining_speed_m_per_s": {"min": 0.460, "max": 0.460},
        "joint_length_m": 27.5,
        "gas_flow_kg_per_s": {"min": 0.123889, "max": 0.123889},
        "time_m": {"min": 50/60, "max": 1}
    },   }


# Generate SoA_data from joining parameters
SoA_data = {}
for product_name in SoA_joining_params:
    process_type = (
        "controlled atmosphere brazing" if product_name == "Battery cooling plate"
        else "TIG" if product_name == "Fuel tank"
        else "standard laser"
    )
    SoA_data[(SoA_NAME, product_name)] = {
        "name": f"{product_name} joined with {process_type}",
        "unit": "unit",
        "location": "NL",
        "exchanges": create_joining_exchanges(SoA_NAME, SoA_joining_params, product_name, elec.key, gas.key),
    }

In [ ]:
SoA.write(SoA_data)

## Use case unit processes with Dynamic Beam Shaping

The functional unit is fixed at `1 joined unit` (`1 unit`).

In [ ]:
DBW_NAME = "Dynamic_Beam_Shaping"
DBW = bd.Database(DBW_NAME)

# Joining parameters with min/max ranges for power and speed
# power_kW_joining is the joining (laser) power; power_kW_station is the welding station power (incl. chiller).
dbw_joining_params = {
    "Heat exchanger": {
        "power_kW_joining": {"min": 1.26, "max": 1.35},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.025, "max": 0.025},
        "joint_length_m": 42.9,
        "gas_flow_kg_per_s": {"min": 0.002676, "max": 0.002676},
        "time_m": {"min": 49.5, "max": 49.5}
    },
    "Fuel tank": {
        "power_kW_joining": {"min": 1.62, "max": 2.268},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.025, "max": 0.025},
        "joint_length_m": 5.06,
        "gas_flow_kg_per_s": {"min": 0.000595, "max": 0.000595},
        "time_m": {"min": 18.0, "max": 18.0}
    },
    "Heterogeneous pipe": {
        "power_kW_joining": {"min": 1.26, "max": 1.89},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.025, "max": 0.025},
        "joint_length_m": 0.094,
        "gas_flow_kg_per_s": {"min": 0.000595, "max": 0.000595},
        "time_m": {"min": 0.45, "max": 0.45}
    },
    "Freeze Dryers Shelf": {
        "power_kW_joining": {"min": 1.62, "max": 1.8},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.1, "max": 0.1},
        "joint_length_m": 13.2,
        "gas_flow_kg_per_s": {"min": 0.000595, "max": 0.000595},
        "time_m": {"min": 9.0, "max": 9.0}
    },
    "Battery cooling plate": {
        "power_kW_joining": {"min": 8, "max": 10},  # Range around original
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joint_length_m": 27.5,
        "joining_speed_m_per_s": {"min": 0.4, "max": 0.4},  # Range around original 0.01
        "gas_flow_kg_per_s": {"min": 0, "max": 0},
        "time_m": {"min": 50/60, "max": 1}
    },
}

# Generate DBW_data from joining parameters
DBW_data = {}
for product_name in dbw_joining_params:
    process_type = ("Dynamic Beam Shaping"
    )
    DBW_data[(DBW_NAME, product_name)] = {
        "name": f"{product_name} joined with {process_type}",
        "unit": "unit",
        "location": "NL",
        "exchanges": create_joining_exchanges(DBW_NAME, dbw_joining_params, product_name, elec.key, gas.key),
    }
    print(DBW_data)
    

In [ ]:
DBW.write(DBW_data)


## LCIA results

In [ ]:
import hashlib
import inspect
import json
import pickle
from pathlib import Path

def _compute_lcia_scenarios(process_databases, ef31_methods, cost_factors):
    scenario_results = []
    category_units = {}

    scenarios = ["minimum", "amount", "maximum"]
    scenario_keys = {
        "minimum": ["minimum", "min"],
        "amount": ["amount"],
        "maximum": ["maximum", "max"]
    }

    for database in process_databases:
        for process in database:
            process_exchanges = list(process.exchanges())

            for scenario in scenarios:
                original_amounts = [
                    (exchange, exchange["amount"])
                    for exchange in process_exchanges
                ]

                try:
                    for exchange, original_amount in original_amounts:
                        scenario_amount = original_amount
                        for key in scenario_keys[scenario]:
                            if key in exchange:
                                scenario_amount = exchange[key]
                                break

                        if scenario_amount != original_amount:
                            exchange["amount"] = scenario_amount
                            exchange.save()

                    # Calculate LCIA
                    lca = LCA({process: 1}, method=ef31_methods[0])
                    lca.lci()
                    lca.lcia()

                    # Calculate foreground electricity consumption and total cost
                    electricity_kwh = sum(
                        exchange["amount"]
                        for exchange in process_exchanges
                        if exchange["type"] == "technosphere"
                        and "electricity" in str(exchange.input).lower()
                    )
                    opex = calculate_process_cost(process, scenario, cost_factors)

                    for method in ef31_methods:
                        lca.switch_method(method)
                        lca.lcia()

                        method_tuple = str(method).replace("'", '').strip("()")
                        method_parts = [p.strip() for p in method_tuple.split(",")]
                        impact_category = method_parts[2] if len(method_parts) > 2 else str(method)

                        category_units[impact_category] = bd.Method(method).metadata.get("unit", "")

                        scenario_results.append({
                            "Database": database.filename,
                            "Process": process["name"],
                            "Key": process.key[1],
                            "Scenario": scenario,
                            "Method": str(method),
                            "Impact Category": impact_category,
                            "Amount": lca.score,
                            "Electricity Consumption (kWh)": electricity_kwh,
                            "Consumable costs (EUR)": opex
                        })
                finally:
                    for exchange, original_amount in original_amounts:
                        if exchange["amount"] != original_amount:
                            exchange["amount"] = original_amount
                            exchange.save()

    return scenario_results, category_units


# Set to True (or pass force_recalc=True) to ignore the cache and recompute the LCIA
FORCE_RECALC = False
LCIA_CACHE_DIR = Path("results/.cache")


def _lcia_cache_key(process_databases, ef31_methods, cost_factors):
    """Hash everything the LCIA results depend on, so edits invalidate the cache."""
    parts = [ECOINVENT_DATABASE_NAME, repr(sorted(cost_factors.items())), repr(list(ef31_methods))]
    try:
        parts.append(inspect.getsource(calculate_process_cost))
    except (OSError, TypeError):
        pass
    dump = lambda obj: json.dumps(dict(obj._data), sort_keys=True, default=str)
    for database in process_databases:
        # Database iteration order is not stable, so sort before hashing
        for process in sorted(database, key=lambda a: a.key):
            parts.append(dump(process))
            parts.extend(sorted(dump(exchange) for exchange in process.exchanges()))
    return hashlib.sha256("\n".join(parts).encode()).hexdigest()[:16]


def calculate_lcia_scenarios(process_databases, ef31_methods, cost_factors=None, force_recalc=None):
    if cost_factors is None:
        cost_factors = COST_FACTORS
    if force_recalc is None:
        force_recalc = FORCE_RECALC

    if not isinstance(process_databases, (list, tuple)):
        process_databases = [process_databases]

    LCIA_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache_file = LCIA_CACHE_DIR / f"lcia_{_lcia_cache_key(process_databases, ef31_methods, cost_factors)}.pkl"

    if cache_file.exists() and not force_recalc:
        with open(cache_file, "rb") as f:
            scenario_results, category_units = pickle.load(f)
        print(f"Loaded cached LCIA results from {cache_file} (set FORCE_RECALC = True to recompute)")
    else:
        scenario_results, category_units = _compute_lcia_scenarios(
            process_databases, ef31_methods, cost_factors
        )
        with open(cache_file, "wb") as f:
            pickle.dump((scenario_results, category_units), f)
        print(f"Computed LCIA results and cached them in {cache_file}")

    # Create DataFrame and add costs
    scenario_df = pd.DataFrame(scenario_results)

    pivot_scenario_df = scenario_df.pivot_table(
        index=["Database", "Process", "Key", "Scenario"],
        columns="Impact Category",
        values="Amount",
        aggfunc="sum"
    ).reset_index()

    result_index = ["Database", "Process", "Key", "Scenario"]
    resource_df = scenario_df[result_index + ["Electricity Consumption (kWh)", "Consumable costs (EUR)"]].drop_duplicates(subset=result_index)

    final_df = pd.merge(pivot_scenario_df, resource_df,
                       on=result_index, how="left")

    # Column headers with units for display and export; final_df keeps plain names
    export_df = final_df.rename(columns={
        cat: f"{cat} [{unit}]" for cat, unit in category_units.items() if unit
    })

    print("\n\n=== EF 3.1 LCIA Results with Foreground Electricity Consumption and Costs ===")
    print(export_df.to_markdown(index=False, floatfmt=".4e"))

    # Save results as CSV and JSON
    if len(process_databases) == 1:
        output_file = f"{process_databases[0].filename.split('.')[0]}_LCIA_results.csv"
    else:
        db_names = "_".join([db.filename.split('.')[0] for db in process_databases])
        output_file = f"results/{db_names}_LCIA_results.csv"

    json_file = output_file.rsplit(".", 1)[0] + ".json"

    export_df.to_csv(output_file, index=False)
    export_df.to_json(json_file, orient="records", indent=2)

    print(f"\nScenario results saved to {output_file} and {json_file}")

    return final_df

In [ ]:
# Capture LCIA results (re-uses calculate_lcia_scenarios output)
lcia_df = calculate_lcia_scenarios([SoA, DBW], ef31_methods)

In [ ]:
import textwrap
import ipywidgets as widgets
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

use_cases = list(SoA_joining_params.keys())
result_index = ["Database", "Process", "Key", "Scenario"]
indicator_cols = [c for c in lcia_df.columns if c not in result_index]

all_technologies = set()
for db_name in [SoA_NAME, DBW_NAME]:
    db_data = lcia_df[lcia_df["Database"].str.startswith(db_name)]
    for process_name in db_data["Process"].unique():
        all_technologies.add(process_name.split(" joined with ", 1)[-1])

technologies = sorted(all_technologies)
tech_color_map = {
    tech: px.colors.qualitative.D3[i % 10]
    for i, tech in enumerate(technologies)
}

wrap_label = lambda tech: "<br>".join(textwrap.wrap(tech, 16))   # long names break onto several lines

def bar_stats(use_case, db_name, indicator):
    """Return (technology, amount, low, high) for one bar, or None if data is missing."""
    rows = lcia_df[(lcia_df["Key"] == use_case) & lcia_df["Database"].str.startswith(db_name)]
    if rows.empty:
        return None
    tech = rows.iloc[0]["Process"].split(" joined with ", 1)[-1]
    val = lambda s: rows.loc[rows["Scenario"] == s, indicator]
    if val("amount").empty or val("minimum").empty or val("maximum").empty:
        return tech, None, None, None
    amount, vmin, vmax = float(val("amount").iloc[0]), float(val("minimum").iloc[0]), float(val("maximum").iloc[0])
    return tech, amount, min(vmin, vmax, amount), max(vmin, vmax, amount)

# One bar trace per (use case, database); values are filled in by update_plot
n_cols = 3
n_rows = -(-len(use_cases) // n_cols)
bar_fig = go.FigureWidget(make_subplots(
    rows=n_rows, cols=n_cols, subplot_titles=use_cases,
    vertical_spacing=0.2, horizontal_spacing=0.08))

legend_shown = set()
bar_slots = []   # (trace index, use case, database name)
for i, use_case in enumerate(use_cases):
    for db_name in [SoA_NAME, DBW_NAME]:
        stats = bar_stats(use_case, db_name, indicator_cols[0])
        tech = stats[0] if stats else ""
        bar_fig.add_trace(go.Bar(
            x=[wrap_label(tech)], y=[0], width=0.55, name=tech,
            marker_color=tech_color_map.get(tech),
            legendgroup=tech, showlegend=tech not in legend_shown,
            error_y=dict(type="data", symmetric=False, array=[0], arrayminus=[0], thickness=1.5, width=5),
            customdata=[[0, 0]],
            hovertemplate="<b>%{fullData.name}</b><br>Value: %{y:.4g}<br>Min: %{customdata[0]:.4g}<br>Max: %{customdata[1]:.4g}<extra></extra>",
        ), row=i // n_cols + 1, col=i % n_cols + 1)
        legend_shown.add(tech)
        bar_slots.append((len(bar_fig.data) - 1, use_case, db_name))

bar_fig.update_layout(
    template="plotly_white", height=330 * n_rows + 120, width=1000, bargap=0.3,
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.08, yanchor="top"),
    margin=dict(l=70, r=30, t=100, b=90), font=dict(size=12))
bar_fig.update_yaxes(gridcolor="#e6e9ec", title_font_size=10, exponentformat="e")
bar_fig.update_xaxes(tickfont_size=10, tickangle=0)   # keep tech names horizontal
bar_fig.update_annotations(font_size=12)   # subplot titles

def update_plot(indicator):
    unit = method_units.get(indicator, "unit")
    with bar_fig.batch_update():
        for idx, use_case, db_name in bar_slots:
            stats = bar_stats(use_case, db_name, indicator)
            if stats is None or stats[1] is None:
                bar_fig.data[idx].y = [None]
                continue
            tech, amount, low, high = stats
            trace = bar_fig.data[idx]
            trace.y = [amount]
            trace.error_y.array = [high - amount]
            trace.error_y.arrayminus = [amount - low]
            trace.customdata = [[low, high]]
        bar_fig.update_yaxes(title_text=unit)
        bar_fig.update_layout(title=dict(text=f"<b>{indicator}</b>", x=0.5, xanchor="center", y=0.98, yanchor="top", font=dict(size=16)))

indicator_dropdown = widgets.Dropdown(
    options=indicator_cols,
    description="Indicator:",
    layout=widgets.Layout(width="400px"),
    style={"description_width": "initial"},
)
indicator_dropdown.observe(lambda change: update_plot(change["new"]), names="value")
update_plot(indicator_dropdown.value)
display(widgets.VBox([indicator_dropdown, bar_fig]))


## Regional analysis: EU electricity markets
Replaces the RER electricity market with each member state's own ecoinvent market and maps the resulting impacts per joined unit.

In [ ]:
# Impact per kWh of each EU member state's own electricity market (and of the argon input)
EU27 = {"AT":"AUT","BE":"BEL","BG":"BGR","HR":"HRV","CY":"CYP","CZ":"CZE","DK":"DNK","EE":"EST","FI":"FIN",
        "FR":"FRA","DE":"DEU","GR":"GRC","HU":"HUN","IE":"IRL","IT":"ITA","LV":"LVA","LT":"LTU","LU":"LUX",
        "MT":"MLT","NL":"NLD","PL":"POL","PT":"PRT","RO":"ROU","SK":"SVK","SI":"SVN","ES":"ESP","SE":"SWE"}

def unit_impacts(act):
    lca = LCA({act: 1}, method=ef31_methods[0]); lca.lci()
    out = {}
    for m in ef31_methods:
        lca.switch_method(m); lca.lcia(); out[m[2]] = lca.score
    return out

def _compute_grid():
    grid = {"RER": unit_impacts(elec)}   # baseline used above
    for iso2 in EU27:
        market = next(a for a in ei if a["name"] == "market for electricity, medium voltage" and a["location"] == iso2)
        grid[iso2] = unit_impacts(market)
    return grid, unit_impacts(gas)

# Cached like calculate_lcia_scenarios (same FORCE_RECALC flag and cache folder)
_grid_key = hashlib.sha256(repr((ECOINVENT_DATABASE_NAME, elec.key, gas.key, sorted(EU27), list(ef31_methods))).encode()).hexdigest()[:16]
_grid_cache = LCIA_CACHE_DIR / f"grid_{_grid_key}.pkl"
LCIA_CACHE_DIR.mkdir(parents=True, exist_ok=True)
if _grid_cache.exists() and not FORCE_RECALC:
    with open(_grid_cache, "rb") as f:
        grid, gas_imp = pickle.load(f)
    print(f"Loaded cached grid impacts from {_grid_cache} (set FORCE_RECALC = True to recompute)")
else:
    grid, gas_imp = _compute_grid()
    with open(_grid_cache, "wb") as f:
        pickle.dump((grid, gas_imp), f)
    print(f"Computed grid impacts and cached them in {_grid_cache}")

# Industrial electricity prices by country, EUR/kWh (Eurostat nrg_pc_205, 2025-S1, band ID 2-20 GWh/yr,
# excl. VAT and recoverable taxes). "RER" uses the EU27 average. Argon price stays at COST_FACTORS.
ELEC_PRICE = {"BE": 0.1659, "BG": 0.1206, "CZ": 0.1655, "DK": 0.126, "DE": 0.1932, "EE": 0.1302, "IE": 0.2357, "GR": 0.1811, "ES": 0.125, "FR": 0.1296, "HR": 0.1534, "IT": 0.203, "CY": 0.2267, "LV": 0.113, "LT": 0.1347, "LU": 0.1476, "HU": 0.1871, "MT": 0.1177, "NL": 0.1889, "AT": 0.1803, "PL": 0.1726, "PT": 0.123, "RO": 0.1633, "SI": 0.1456, "SK": 0.1782, "FI": 0.0697, "SE": 0.0838,
              "RER": 0.164}

# Foreground processes only have electricity + gas inputs, so the score is linear in both
rows = []
for db in (SoA, DBW):
    for proc in db:
        tech = proc["name"].split(" joined with ")[-1]
        kwh = sum(e["amount"] for e in proc.technosphere() if "electricity" in e.input["name"])
        kg  = sum(e["amount"] for e in proc.technosphere() if "electricity" not in e.input["name"])
        for c, imp in grid.items():
            for ind in imp:
                rows.append({"use_case": proc.key[1], "technology": tech, "country": c, "indicator": ind,
                             "impact": kwh * imp[ind] + kg * gas_imp[ind]})
            # Consumable costs: country-specific electricity price + argon at the notebook's price
            rows.append({"use_case": proc.key[1], "technology": tech, "country": c, "indicator": "Consumable costs (EUR)",
                         "impact": kwh * ELEC_PRICE[c] + kg * COST_FACTORS["argon_EUR_per_kg"]})
regional_df = pd.DataFrame(rows)
regional_df.to_csv("results/regional_joining.csv", index=False)   # read by workflow_assembly.ipynb
regional_df.head()


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ind_opts = [m[2] for m in ef31_methods] + ["Consumable costs (EUR)"]
method_units = {**method_units, "Consumable costs (EUR)": "EUR"}
codes = list(EU27)
iso3  = [EU27[c] for c in codes]
short = lambda s: s.split(" (EF")[0]          # drop the "(EF v3.1 - IPCC 2021)" suffix in titles

wide = widgets.Layout(width="420px")
sty  = {"description_width": "90px"}
w_a    = widgets.Dropdown(options=ind_opts, value=ind_opts[0],  description="Indicator A", layout=wide, style=sty)
w_b    = widgets.Dropdown(options=ind_opts, value=ind_opts[-1], description="Indicator B", layout=wide, style=sty)
w_uc   = widgets.Dropdown(options=list(SoA_joining_params), description="Use case", layout=wide, style=sty)
w_tech = widgets.Dropdown(options=["State of the Art", "Dynamic Beam Shaping"], description="Technology", layout=wide, style=sty)
w_view = widgets.Dropdown(options=["Absolute", "Change vs RER (%)"], description="View", layout=wide, style=sty)

# Two maps (A, B) on top and a country-by-country scatter of A vs B below
fig = go.FigureWidget(make_subplots(
    rows=2, cols=2, specs=[[{"type": "choropleth"}, {"type": "choropleth"}], [{"type": "xy", "colspan": 2}, None]],
    row_heights=[0.6, 0.4], vertical_spacing=0.1, horizontal_spacing=0.04, subplot_titles=("A", "B", "")))
cbar = lambda x: dict(x=x, y=0.79, len=0.42, thickness=12, outlinewidth=0, tickfont=dict(size=10), title=dict(font=dict(size=11)))
hover = "<b>%{text}</b><br>%{z:.4g}<extra></extra>"
fig.add_trace(go.Choropleth(locations=iso3, z=[0] * len(codes), text=codes, hovertemplate=hover, colorbar=cbar(0.455),
                            marker=dict(line=dict(color="white", width=0.5))), row=1, col=1)
fig.add_trace(go.Choropleth(locations=iso3, z=[0] * len(codes), text=codes, hovertemplate=hover, colorbar=cbar(1.0),
                            marker=dict(line=dict(color="white", width=0.5))), row=1, col=2)
fig.add_trace(go.Scatter(x=[0] * len(codes), y=[0] * len(codes), mode="markers+text", text=codes, textposition="top center",
                         textfont=dict(size=10), marker=dict(size=9, color="#1f77b4", line=dict(color="white", width=1)),
                         hovertemplate="<b>%{text}</b><br>A: %{x:.4g}<br>B: %{y:.4g}<extra></extra>"), row=2, col=1)
fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers+text", text=["RER"], textposition="bottom center", textfont=dict(size=10),
                         marker=dict(size=13, symbol="diamond", color="#d62728", line=dict(color="white", width=1)),
                         hovertemplate="<b>RER baseline</b><br>A: %{x:.4g}<br>B: %{y:.4g}<extra></extra>"), row=2, col=1)
fig.update_geos(scope="europe", fitbounds="locations", showframe=False, showcountries=True,
                countrycolor="#c4cad0", showland=True, landcolor="#eceff1", showocean=False, showlakes=False,
                bgcolor="rgba(0,0,0,0)")
fig.update_layout(template="plotly_white", height=950, showlegend=False, font=dict(size=12),
                  margin=dict(l=60, r=30, t=60, b=50))
fig.update_annotations(font_size=13)
fig.update_xaxes(zeroline=False, gridcolor="#e6e9ec"); fig.update_yaxes(zeroline=False, gridcolor="#e6e9ec")

def series(ind):
    d = regional_df[(regional_df.use_case == w_uc.value) & (regional_df.indicator == ind)]
    is_dbw = d.technology == "Dynamic Beam Shaping"
    d = d[is_dbw if w_tech.value == "Dynamic Beam Shaping" else ~is_dbw].set_index("country")["impact"]
    if w_view.value == "Absolute":
        return d[codes].values, d["RER"], method_units.get(ind, "")
    return ((d[codes] / d["RER"] - 1) * 100).values, 0.0, "% vs RER"

def update(_=None):
    (za, ra, ua), (zb, rb, ub) = series(w_a.value), series(w_b.value)
    pct = w_view.value != "Absolute"
    with fig.batch_update():
        for trace, z, u, scale in ((fig.data[0], za, ua, "YlOrRd"), (fig.data[1], zb, ub, "GnBu")):
            m = max(abs(z).max(), 1e-12)
            trace.z = z
            trace.colorscale = "RdBu_r" if pct else scale
            trace.zmin, trace.zmax = (-m, m) if pct else (None, None)
            trace.colorbar.title.text = u
        fig.data[2].x, fig.data[2].y = za, zb
        fig.data[3].x, fig.data[3].y = [ra], [rb]
        # explicit ranges (incl. the RER point) so the points can't end up outside an un-updated autorange
        for axis, vals, ref in ((fig.layout.xaxis, za, ra), (fig.layout.yaxis, zb, rb)):
            lo, hi = min(vals.min(), ref), max(vals.max(), ref)
            pad = (hi - lo) * 0.08 or abs(hi) * 0.1 or 1.0
            axis.range = [lo - pad, hi + pad]
        fig.layout.annotations[0].text = f"<b>A</b>  {short(w_a.value)}"
        fig.layout.annotations[1].text = f"<b>B</b>  {short(w_b.value)}"
        fig.layout.xaxis.title.text = f"A: {short(w_a.value)} [{ua}]"
        fig.layout.yaxis.title.text = f"B: {short(w_b.value)} [{ub}]"
        fig.layout.title = dict(text=f"{w_uc.value}, {w_tech.value}: per joined unit", x=0.01, font=dict(size=15))

for w in (w_a, w_b, w_uc, w_tech, w_view): w.observe(update, names="value")
update()
display(widgets.VBox([widgets.HBox([w_a, w_b]), widgets.HBox([w_uc, w_tech]), w_view, fig]))
